In [6]:
import os
import urllib.request
import time
import akshare as ak
import pandas as pd

# ==========================================
# 终极网络环境清理（屏蔽所有系统代理）
# ==========================================
os.environ["http_proxy"] = ""
os.environ["https_proxy"] = ""
os.environ["HTTP_PROXY"] = ""
os.environ["HTTPS_PROXY"] = ""
os.environ["all_proxy"] = ""
os.environ["ALL_PROXY"] = ""
# 暴力覆盖底层获取代理的函数，让 requests 彻底变成“瞎子”，只能直连
urllib.request.getproxies = lambda: {}

def get_target_stocks_akshare():
    print("正在通过 AkShare 获取全市场 A 股最新实时/盘后数据...")

    # ==========================================
    # 增加容错与重试机制
    # ==========================================
    max_retries = 3
    df = pd.DataFrame()
    for attempt in range(max_retries):
        try:
            # 获取东方财富的实时 A 股数据
            df = ak.stock_zh_a_spot()
            print("✅ 全市场数据拉取成功！")
            break  # 成功拉取则跳出循环
        except Exception as e:
            print(f"⚠️ 第 {attempt + 1} 次尝试拉取失败 (原因: 服务器断开连接)。")
            if attempt < max_retries - 1:
                print("⏳ 等待 3 秒后重试...")
                time.sleep(3)
            else:
                print("❌ 已达到最大重试次数，请稍后再试或检查网络状态。")
                return pd.DataFrame()

    if df.empty:
        return df

    print(f"共获取到 {len(df)} 只股票，开始进行清洗和过滤...")

    # 1. 过滤掉停牌股票
    df = df[df['成交量'] > 0].copy()
    df = df.dropna(subset=['最新价', '市盈率-动态', '市净率', '总市值'])

    # 2. 过滤非 ST 股票
    df = df[~df['名称'].str.contains('ST')]

    # 3. 过滤市盈率 > 0 且 市净率 > 0
    df = df[(df['市盈率-动态'] > 0) & (df['市净率'] > 0)]

    # 4. 过滤沪深主板股票（仅保留 60 和 00 开头的股票）
    main_board_mask = df['代码'].str.contains(r'^(60|00)', regex=True)
    df = df[main_board_mask]

    if df.empty:
        print("没有符合条件的股票。")
        return df

    # 5. 按市值从小到大排序，并截取前 50 只
    top_50_smallest_cap = df.sort_values(by='总市值', ascending=True).head(50)

    # 6. 在这 50 只股票的基础上，按价格由低到高排序
    final_result = top_50_smallest_cap.sort_values(by='最新价', ascending=True)

    # 重置索引并格式化输出
    final_result = final_result.reset_index(drop=True)

    # 将市值转换为“亿元”单位，保留两位小数，方便阅读
    final_result['总市值(亿)'] = (final_result['总市值'] / 100000000).round(2)
    final_result['流通市值(亿)'] = (final_result['流通市值'] / 100000000).round(2)

    # 选取需要展示的核心列
    columns_to_show = ['代码', '名称', '最新价', '市盈率-动态', '市净率', '总市值(亿)', '流通市值(亿)']

    return final_result[columns_to_show]

if __name__ == "__main__":
    result_df = get_target_stocks_akshare()

    if not result_df.empty:
        print("\n=== 最终筛选结果（主板中市值最小的前50只，并按最新价由低到高排序） ===")
        pd.set_option('display.max_rows', 50)
        print(result_df)

正在通过 AkShare 获取全市场 A 股最新实时/盘后数据...


Please wait for a moment:   0%|          | 0/69 [00:00<?, ?it/s]

✅ 全市场数据拉取成功！
共获取到 5496 只股票，开始进行清洗和过滤...


KeyError: ['市盈率-动态', '市净率', '总市值']

In [3]:
import akshare as ak
import pandas as pd

  0%|          | 0/58 [00:00<?, ?it/s]

ProxyError: HTTPSConnectionPool(host='82.push2.eastmoney.com', port=443): Max retries exceeded with url: /api/qt/clist/get?pn=3&pz=100&po=1&np=1&ut=bd1d9ddb04089700cf9c27f6f7426281&fltt=2&invt=2&fid=f12&fs=m%3A0+t%3A6%2Cm%3A0+t%3A80%2Cm%3A1+t%3A2%2Cm%3A1+t%3A23%2Cm%3A0+t%3A81+s%3A2048&fields=f1%2Cf2%2Cf3%2Cf4%2Cf5%2Cf6%2Cf7%2Cf8%2Cf9%2Cf10%2Cf12%2Cf13%2Cf14%2Cf15%2Cf16%2Cf17%2Cf18%2Cf20%2Cf21%2Cf23%2Cf24%2Cf25%2Cf22%2Cf11%2Cf62%2Cf128%2Cf136%2Cf115%2Cf152 (Caused by ProxyError('Unable to connect to proxy', RemoteDisconnected('Remote end closed connection without response')))